# Soluciones — Clase 23: Reducción de dimensionalidad — PCA

> **Nota para el docente:** Este notebook contiene soluciones completas con comentarios de corrección. No lo compartas con los alumnos antes de la entrega.

In [ ]:
# Importaciones completas para todas las soluciones
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

plt.rcParams['figure.figsize'] = (9, 5)
print('Librerías cargadas ✅')

## Solución — Preparación de datos y correlación

In [ ]:
# Generar dataset simulado para la solución
# 5 variables correlacionadas que representan el rendimiento estudiantil
np.random.seed(42)
n = 250

# Factor latente: habilidad general del estudiante
habilidad = np.random.randn(n)

df = pd.DataFrame({
    'nota_matematicas': np.clip(65 + habilidad * 14 + np.random.randn(n) * 5, 0, 100),
    'nota_ciencias':    np.clip(63 + habilidad * 13 + np.random.randn(n) * 6, 0, 100),
    'nota_lengua':      np.clip(67 + habilidad * 9  + np.random.randn(n) * 7, 0, 100),
    'horas_estudio':    np.clip(3.5 + habilidad * 1.2 + np.random.randn(n) * 0.4, 0, 10),
    'faltas':           np.clip(4 - habilidad * 1.8 + np.random.randn(n) * 1, 0, 20)
})

cols_num = df.columns.tolist()
X = df[cols_num].dropna()

# Correlación
corr = X.corr().round(2)
print('Matriz de correlación:')
print(corr)
print('\n► notas y horas_estudio tienen correlación positiva fuerte')
print('► faltas tiene correlación NEGATIVA con notas (más faltas = peores notas)')

## Solución — Escalar y Scree Plot

In [ ]:
# Escalar: paso OBLIGATORIO antes de PCA
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# PCA completo para ver toda la varianza
pca_full = PCA()
pca_full.fit(X_scaled)

varianza_ind  = pca_full.explained_variance_ratio_ * 100
varianza_acum = varianza_ind.cumsum()
n_comp = len(varianza_ind)

print('Varianza explicada:')
for i, (v, a) in enumerate(zip(varianza_ind, varianza_acum)):
    marca = ''
    if a >= 80 and (i == 0 or varianza_acum[i-1] < 80):
        marca = ' ← cruzamos 80%'
    if a >= 95 and (i == 0 or varianza_acum[i-1] < 95):
        marca = ' ← cruzamos 95%'
    print(f'  PC{i+1}: {v:.1f}%  (acum: {a:.1f}%){marca}')

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
ejes = range(1, n_comp+1)

axes[0].bar(ejes, varianza_ind, color='#1976D2', edgecolor='k')
axes[0].set_xlabel('Componente')
axes[0].set_ylabel('Varianza (%)')
axes[0].set_title('Varianza individual')

axes[1].plot(ejes, varianza_acum, marker='o', color='coral', linewidth=2)
axes[1].fill_between(ejes, varianza_acum, alpha=0.15, color='coral')
axes[1].axhline(80, color='green', linestyle='--', label='80%')
axes[1].axhline(95, color='red',   linestyle='--', label='95%')
axes[1].set_ylim(0, 105)
axes[1].set_xlabel('Número de componentes')
axes[1].set_ylabel('Varianza acumulada (%)')
axes[1].set_title('Scree Plot')
axes[1].legend()

plt.tight_layout()
plt.show()

# Con 5 variables muy correlacionadas, 2 componentes suelen explicar >80%
n_para_80 = next(i+1 for i, a in enumerate(varianza_acum) if a >= 80)
print(f'\n► Necesitas {n_para_80} componente(s) para explicar ≥80% de la varianza')

## Solución — PCA a 2 dimensiones, Scatter y Biplot

In [ ]:
# PCA final con 2 componentes
pca = PCA(n_components=2)
X_pca = pca.fit_transform(X_scaled)

var1 = pca.explained_variance_ratio_[0] * 100
var2 = pca.explained_variance_ratio_[1] * 100

print(f'PC1: {var1:.1f}%  |  PC2: {var2:.1f}%  |  Total: {var1+var2:.1f}%')

# Loadings: contribución de cada variable a cada componente
loadings = pca.components_.T
df_loadings = pd.DataFrame(loadings, index=cols_num, columns=['PC1', 'PC2'])
print('\nLoadings (contribución de cada variable):')
print(df_loadings.round(3))
print('\n► PC1 positivo: estudiante con buenas notas, muchas horas de estudio y pocas faltas')
print('► PC1 puede interpretarse como "rendimiento general"')

In [ ]:
# Biplot completo y bien anotado
loadings = pca.components_.T
escala = 2.2
colores_var = ['#C62828', '#AD1457', '#6A1B9A', '#1565C0', '#2E7D32']

fig, ax = plt.subplots(figsize=(9, 7))

ax.scatter(X_pca[:, 0], X_pca[:, 1],
           alpha=0.3, s=25, color='#90CAF9', label='Estudiantes')

for i, nombre in enumerate(cols_num):
    lx = loadings[i, 0] * escala
    ly = loadings[i, 1] * escala
    c  = colores_var[i]
    ax.annotate(
        '', xy=(lx, ly), xytext=(0, 0),
        arrowprops=dict(arrowstyle='->', color=c, lw=2)
    )
    ax.text(lx * 1.12, ly * 1.12, nombre,
            color=c, fontsize=10, fontweight='bold', ha='center')

ax.axhline(0, color='gray', lw=0.5, ls='--')
ax.axvline(0, color='gray', lw=0.5, ls='--')
ax.set_xlabel(f'PC1 ({var1:.1f}% varianza)', fontsize=12)
ax.set_ylabel(f'PC2 ({var2:.1f}% varianza)', fontsize=12)
ax.set_title('Biplot PCA — Dataset Estudiantes', fontsize=13)
plt.tight_layout()
plt.show()

print('Interpretación del biplot:')
print('  - Las flechas de notas y horas_estudio apuntan en la misma dirección → correlación positiva')
print('  - La flecha de faltas apunta en dirección opuesta → correlación negativa con notas')
print('  - PC1 separa estudiantes de alto vs bajo rendimiento')

## Solución — PCA + Clustering con interpretación

In [ ]:
# K-Means en espacio PCA
kmeans = KMeans(n_clusters=3, random_state=42, n_init=10)
etiquetas = kmeans.fit_predict(X_pca)

score = silhouette_score(X_pca, etiquetas)
print(f'Silhouette Score: {score:.3f}')

# Visualización
paleta = ['#1976D2', '#D32F2F', '#388E3C']
nombres = ['Rendimiento bajo', 'Rendimiento alto', 'Rendimiento medio']

fig, ax = plt.subplots(figsize=(9, 6))
for i in range(3):
    mask = etiquetas == i
    ax.scatter(X_pca[mask, 0], X_pca[mask, 1],
               c=paleta[i], alpha=0.7, s=50, edgecolors='k', lw=0.3,
               label=f'Cluster {i}')

centros = kmeans.cluster_centers_
ax.scatter(centros[:, 0], centros[:, 1],
           marker='X', s=300, color='black', zorder=10, label='Centroides')

ax.set_xlabel(f'PC1 ({var1:.1f}%)')
ax.set_ylabel(f'PC2 ({var2:.1f}%)')
ax.set_title(f'K-Means (K=3) en espacio PCA  |  Silhouette: {score:.3f}')
ax.legend()
plt.tight_layout()
plt.show()

# Perfil en variables originales
df_res = X.copy()
df_res['cluster'] = etiquetas
perfil = df_res.groupby('cluster').mean().round(2)
conteo = df_res['cluster'].value_counts().sort_index()

print('\nPerfil promedio por cluster (en variables ORIGINALES):')
print(perfil.join(conteo.rename('n_estudiantes')))

print('\n► El cluster con nota_matematicas más alta = alto rendimiento')
print('► El cluster con más faltas = bajo rendimiento')
print('► El cluster intermedio = potencial de mejora')